# lawgraph validation suite
Set `ROOT` (the directory containing `lawgraph/`) and `DB`. Each section states what a correct result looks like. Cluster level (one node per decision) is the default; opinion level is checked where it matters.

In [ ]:
import sys, datetime as dt, duckdb, pandas as pd, numpy as np
ROOT = "."                 # <-- directory that contains lawgraph/
DB   = "graph.duckdb"      # <-- your file
sys.path.insert(0, ROOT)
from lawgraph.store import Store
from lawgraph.frame import Frame, timeline
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 60)

st = Store(DB, read_only=True); con = st.con
CL = "https://www.courtlistener.com"
def cl_url(cluster_id): return f"{CL}/opinion/{cluster_id}/x/"
def dk_url(docket_id):  return f"{CL}/docket/{docket_id}/x/"
def ps_url(person_id):  return f"{CL}/person/{person_id}/x/"
def one(sql): return con.execute(sql).fetchall()[0]

## 0. Tables and row counts
**Expect:** 7 node tables, 16 edge tables, no missing.

In [ ]:
expected = {s.table for s in st.schema.specs}
have = {r[0] for r in con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='main'").fetchall()}
print("missing:", sorted(expected - have) or "none")
pd.Series(st.tables(), name="rows").to_frame()

## A. Time slicing
**Expect:** counts grow monotonically in existence mode; active ≤ existence; a node is absent the day before its date and present on it.

In [ ]:
rows = []
for y in range(1800, 2030, 10):
    s = st.at(y); a = st.at(y, active=True)
    rows.append({"year": y, **{t: len(s.index(t)) for t in ["court", "docket", "cluster", "person"]},
                 "person_active": len(a.index("person")), "docket_active": len(a.index("docket"))})
df = pd.DataFrame(rows).set_index("year")
print("monotone:", all((df[c].diff().dropna() >= 0).all() for c in ["court", "docket", "cluster", "person"]))
print("active <= existence:", (df.person_active <= df.person).all() and (df.docket_active <= df.docket).all())
df

In [ ]:
cid, d = one("SELECT id, valid_from FROM node_cluster WHERE valid_from IS NOT NULL ORDER BY random() LIMIT 1")
flags = [cid in st.at(d - dt.timedelta(days=1)).index("cluster"), cid in st.at(d).index("cluster"), cid in st.at(d + dt.timedelta(days=365)).index("cluster")]
print(f"cluster {cid} filed {d}: day-before={flags[0]} (False)  on-date={flags[1]} (True)  year-later={flags[2]} (True)  ->", "OK" if flags == [False, True, True] else "FAIL")
p, w = len(st.at(1990).index("cluster")), len(st.at(1990, until=1991).index("cluster"))
print(f"point 1990: {p:,}   window [1990,1991): {w:,}   window >= point:", w >= p)

## B. Opinion → cluster structure
**Expect:** ~2% of clusters carry CL's legacy+new duplicate opinion rows but receive ~20% of citations. This is why the cluster level is canonical.

In [ ]:
con.sql("""
WITH per AS (SELECT dst_id AS cluster, count(*) n, bool_or(src_id < 5000000) legacy, bool_or(src_id > 9000000) new
             FROM edge_opinion_in_cluster_cluster GROUP BY 1)
SELECT count(*) AS clusters, count(*) FILTER (WHERE n > 1) AS multi_opinion,
       count(*) FILTER (WHERE legacy AND new) AS legacy_plus_new,
       round(100.0 * count(*) FILTER (WHERE legacy AND new) / count(*), 2) AS pct_clusters,
       (SELECT round(100.0 * count(*) FILTER (WHERE o.cluster_id IN (SELECT cluster FROM per WHERE legacy AND new)) / count(*), 1)
          FROM edge_opinion_cites_opinion e JOIN node_opinion o ON o.id = e.dst_id) AS pct_of_citations
FROM per""").df()

In [ ]:
# cluster-level cites must equal the opinion-level cites pushed through in_cluster (P^T C P), for a sample of busy clusters
s = st.at(2026)
P = s.adjacency(("opinion", "in_cluster", "cluster")); C = s.adjacency(("opinion", "cites", "opinion"), weight="depth")
K = s.adjacency(("cluster", "cites", "cluster"), weight="depth")
M = (P.T @ C @ P).tocsr(); M.setdiag(0); M.eliminate_zeros()        # the materialised table drops within-cluster self-cites
diff = abs(M - K); print("max |P^T C P - cluster_cites| =", diff.max(), "| nnz difference =", (M != K).nnz, "(expect 0)")

## C. Citations (cluster level)
**Expect:** every famous case found; in-citations non-decreasing from 0; the 2026 count within a few % of CL's own `citation_count`.

In [ ]:
FAMOUS = [("Marbury v. Madison", 1803), ("Brown v. Board", 1954), ("Miranda v. Arizona", 1966), ("Roe v. Wade", 1973),
          ("Chevron", 1984), ("Conley v. Gibson", 1957), ("Bell Atlantic Corp. v. Twombly", 2007),
          ("Ashcroft v. Iqbal", 2009), ("Strickland v. Washington", 1984)]
def find_cluster(name, year):
    r = con.execute(f"""SELECT id, case_name, date_filed, citation_count FROM node_cluster
                        WHERE case_name ILIKE '%{name}%' AND year(date_filed) = {year}
                        ORDER BY citation_count DESC NULLS LAST LIMIT 1""").df()
    return r.iloc[0] if len(r) else None
def cindeg(cid, d):
    return one(f"SELECT count(*) FROM edge_cluster_cites_cluster WHERE dst_id={cid} AND valid_from <= DATE '{d}'")[0]

found, rows = {}, []
for name, year in FAMOUS:
    c = find_cluster(name, year)
    if c is None:
        print(f"{name:<32} NOT FOUND"); continue
    found[name] = c; y0 = c.date_filed.year
    rows.append({"case": name, "cluster": int(c.id), "filed": y0,
                 "+0": cindeg(c.id, f"{y0}-12-31"), "+10": cindeg(c.id, f"{y0+10}-01-01"), "+25": cindeg(c.id, f"{y0+25}-01-01"),
                 "2026": cindeg(c.id, "2026-01-01"), "cl_count": c.citation_count, "url": cl_url(c.id)})
if rows:
    df = pd.DataFrame(rows).set_index("case"); df["ratio"] = (df["2026"] / df["cl_count"]).round(3)
df if rows else print("no famous cases found")

In [ ]:
# most-cited decisions: should be dominated by procedural standards; `ours` vs CL's `citation_count` within ~1%
con.sql("""
SELECT *, round(ours * 1.0 / nullif(cl_count, 0), 3) AS ratio FROM (
  SELECT c.id, c.case_name, c.date_filed, count(*) AS ours, c.citation_count AS cl_count
  FROM edge_cluster_cites_cluster e JOIN node_cluster c ON c.id = e.dst_id
  GROUP BY c.id, c.case_name, c.date_filed, c.citation_count)
ORDER BY ours DESC LIMIT 15""").df().assign(url=lambda d: d.id.map(cl_url))

In [ ]:
# anachronisms (citing decision dated before the cited one) -- CL date noise, expect well under 1%
con.sql("""SELECT count(*) AS edges, count(*) FILTER (WHERE e.valid_from < c.date_filed) AS citing_before_cited,
                  round(100.0 * count(*) FILTER (WHERE e.valid_from < c.date_filed) / count(*), 3) AS pct
           FROM edge_cluster_cites_cluster e JOIN node_cluster c ON c.id = e.dst_id""").df()

In [ ]:
# spot check against CL's 'Cited by' tab: 10 most recent decisions citing Iqbal
name = "Ashcroft v. Iqbal" if "Ashcroft v. Iqbal" in found else next(iter(found)); c = found[name]
print(name, cl_url(c.id))
con.sql(f"""SELECT k.id, k.case_name, k.date_filed, e.depth FROM edge_cluster_cites_cluster e JOIN node_cluster k ON k.id = e.src_id
            WHERE e.dst_id = {c.id} ORDER BY k.date_filed DESC LIMIT 10""").df().assign(url=lambda d: d.id.map(cl_url))

## D. Frame: identity across slices
**Expect:** a decision keeps the same row in every sheet; its column sums equal the direct counts and are monotone.

In [ ]:
f = Frame(st, types=["cluster"])
i = f.pos("cluster", [c.id])[0]
ts = timeline(c.date_filed.year, 2026, "5Y")
col = [int(A[:, i].sum()) for A in f.stack(ts, ("cluster", "cites", "cluster"))]
direct = [cindeg(c.id, t) for t in ts]
print("frame position:", i, "| n =", f.n)
print("stack:  ", col); print("direct: ", direct)
print("identical:", col == direct, "| monotone:", bool(np.all(np.diff(col) >= 0)))

## E. Authorship (cluster level)
**Expect:** a justice's authored decisions span exactly their tenure; most author edges fall inside the author's `sits_on` interval.

In [ ]:
for first, last, tenure in [("Antonin", "Scalia", "1986-2016"), ("Ruth Bader", "Ginsburg", "1993-2020"), ("Sandra Day", "O'Connor", "1981-2006")]:
    r = con.execute(f"""SELECT min(year(e.valid_from)), max(year(e.valid_from)), count(*)
                        FROM edge_person_author_cluster e JOIN node_person p ON p.id = e.src_id
                        WHERE p.name_last = '{last.replace("'", "''")}' AND p.name_first = '{first}'""").fetchall()[0]
    print(f"{first} {last:<10} authored decisions {r[0]}-{r[1]} ({r[2]} total)   expected tenure {tenure}")

In [ ]:
con.sql("""SELECT year(e.valid_from) AS year, count(*) AS decisions
           FROM edge_person_author_cluster e JOIN node_person p ON p.id = e.src_id
           WHERE p.name_last = 'Scalia' AND p.name_first = 'Antonin' GROUP BY 1 ORDER BY 1""").df().set_index("year").T

In [ ]:
con.sql("""
WITH a AS (SELECT e.src_id AS judge, e.valid_from AS filed, d.dst_id AS court
           FROM edge_person_author_cluster e
           JOIN edge_cluster_in_docket_docket cd ON cd.src_id = e.dst_id
           JOIN edge_docket_filed_in_court d ON d.src_id = cd.dst_id)
SELECT count(*) AS author_edges,
       round(100.0 * count(*) FILTER (WHERE EXISTS (SELECT 1 FROM edge_person_sits_on_court s WHERE s.src_id = a.judge AND s.dst_id = a.court
             AND s.active_from <= a.filed AND (s.active_to IS NULL OR s.active_to >= a.filed))) / count(*), 1) AS pct_within_tenure_same_court,
       round(100.0 * count(*) FILTER (WHERE EXISTS (SELECT 1 FROM edge_person_sits_on_court s WHERE s.src_id = a.judge
             AND s.active_from <= a.filed AND (s.active_to IS NULL OR s.active_to >= a.filed))) / count(*), 1) AS pct_within_tenure_any_court
FROM a""").df()

## F. RECAP: parties, attorneys, judge on a docket
**Expect:** the real PACER docket (has `pacer_case_id`) shows the right judge, parties and counsel. Verify on CL.

In [ ]:
pat = "Apple Inc. v. Samsung"
dk = con.execute(f"""SELECT id, court_id, case_name, docket_number, date_filed, date_terminated
                     FROM node_docket WHERE case_name ILIKE '%{pat}%' AND court_id = 'cand'
                       AND pacer_case_id IS NOT NULL AND docket_number LIKE '%:%-cv-%'
                     ORDER BY date_filed LIMIT 5""").df()
dk.assign(url=lambda d: d.id.map(dk_url))

In [ ]:
if len(dk):
    did = int(dk.iloc[0].id); print(dk.iloc[0].case_name, dk.iloc[0].docket_number, dk_url(did))
    print("\njudge(s):");  print(con.sql(f"""SELECT p.name_first, p.name_last, e.role, e.active_from, e.active_to
        FROM edge_person_presides_over_docket e JOIN node_person p ON p.id = e.src_id WHERE e.dst_id = {did}""").df())
    print("\nparties:");   print(con.sql(f"""SELECT pt.name, e.party_type FROM edge_party_party_in_docket e
        JOIN node_party pt ON pt.id = e.src_id WHERE e.dst_id = {did} ORDER BY e.party_type, pt.name LIMIT 20""").df())
    print("\nattorneys:"); print(con.sql(f"""SELECT a.name, e.role_raw, count(*) AS n FROM edge_attorney_attorney_in_docket e
        JOIN node_attorney a ON a.id = e.src_id WHERE e.dst_id = {did} GROUP BY ALL ORDER BY n DESC LIMIT 15""").df())

Known limitation: CL keeps separate scraper-created dockets (no `pacer_case_id`, no dates) that the *opinions* attach to, so `cluster → docket → party` is broken for many cases. `docket_number_core` is the key for stitching them; measured below.

In [ ]:
con.sql("""
SELECT count(*) AS dockets_with_decisions,
       count(*) FILTER (WHERE parties = 0) AS without_parties,
       round(100.0 * count(*) FILTER (WHERE parties = 0) / count(*), 1) AS pct_without_parties
FROM (SELECT cd.dst_id AS docket, (SELECT count(*) FROM edge_party_party_in_docket p WHERE p.dst_id = cd.dst_id) AS parties
      FROM edge_cluster_in_docket_docket cd GROUP BY 1)""").df()

## G. Referential integrity
**Expect:** dangling endpoints a tiny fraction on every relation.

In [ ]:
rows = []
for (s_, rel, d_), spec in st.schema.edges.items():
    sn, dn = st.schema.nodes[s_].table, st.schema.nodes[d_].table
    n, ds, dd = one(f"""SELECT count(*), count(*) FILTER (WHERE a.id IS NULL), count(*) FILTER (WHERE b.id IS NULL)
        FROM {spec.table} e LEFT JOIN {sn} a ON a.id = e.src_id LEFT JOIN {dn} b ON b.id = e.dst_id""")
    rows.append((spec.table, n, ds, dd, round(100 * (ds + dd) / max(n, 1), 3)))
pd.DataFrame(rows, columns=["edge", "rows", "dangling_src", "dangling_dst", "pct"]).set_index("edge")

## H. Random spot-check sheet
Open each link and compare name, date, author, cited-by.

In [ ]:
con.sql("""
SELECT c.id, c.case_name, c.date_filed, c.precedential_status,
       (SELECT count(*) FROM edge_cluster_cites_cluster e WHERE e.src_id = c.id) AS cites_out,
       (SELECT count(*) FROM edge_cluster_cites_cluster e WHERE e.dst_id = c.id) AS cited_by, c.citation_count AS cl_count,
       (SELECT p.name_last FROM edge_person_author_cluster a JOIN node_person p ON p.id = a.src_id WHERE a.dst_id = c.id LIMIT 1) AS author
FROM node_cluster c
WHERE c.date_filed >= DATE '1950-01-01' AND c.precedential_status = 'Published' AND c.citation_count > 20
ORDER BY random() LIMIT 10""").df().assign(url=lambda d: d.id.map(cl_url))

In [ ]:
con.sql("""
SELECT d.id, d.court_id, d.case_name, d.docket_number, d.date_filed, d.date_terminated,
       (SELECT count(*) FROM edge_party_party_in_docket e WHERE e.dst_id = d.id) AS parties,
       (SELECT count(*) FROM edge_attorney_attorney_in_docket e WHERE e.dst_id = d.id) AS attorney_roles,
       (SELECT p.name_last FROM edge_person_presides_over_docket e JOIN node_person p ON p.id = e.src_id
         WHERE e.dst_id = d.id AND e.role = 'assigned' LIMIT 1) AS judge
FROM node_docket d
WHERE d.date_filed >= DATE '2015-01-01' AND d.assigned_to_id IS NOT NULL AND d.pacer_case_id IS NOT NULL
  AND d.docket_number LIKE '%-cv-%'
ORDER BY random() LIMIT 10""").df().assign(url=lambda d: d.id.map(dk_url))